<img src='../OUTILS/bandeau_MF.png' align='right' width='100%'/>

# <div style='background-color: #1e3c72; color: white; padding: 20px; border-radius: 10px; text-align: center;'>🌍🛰️ Manipulation de données satellitaires - Format TIF</div>

---

## 🎯 **Objectifs du TP**

##  🔎 Faire une image pansharpening
##  🔄 Faire une image sandwich

---

## 🧰 **Workflow : lignes de commandes bash (GDAL & ImageMagick)**

Ce TP utilise principalement la bibliothèque **GDAL** (*Geospatial Data Abstraction Library*), un outil puissant pour manipuler les données issues de satellites météorologiques.

Les fichiers utilisés sont au format **TIF**, issus de la production opérationnelle de Météo-France.

---

## 📂 **Fichier utilisé**

Les fichiers TIF étudiés sont des produits RGB et monocanal du satellite Meteosat-12 :

📁 `/stockage/DATA/TIF/20260409/...`

> 💡 *Ce fichier est un produit opérationnel du Centre de Météorologie Spatiale (CMS) de Météo-France.*

<div class="alert alert-info" role="alert">
<h3> ⚙️ Initialisation de l'environnement</h3>
Importation des librairies nécessaires et configuration des chemins d'accès.
</div>

In [ ]:
from datetime import datetime
import sys
import os
from osgeo import gdal
from PIL import Image
import subprocess
from IPython.display import display,HTML
os.environ['PATH'] = f"/opt/conda/env_MF_teledetection/bin:{os.environ['PATH']}" 
os.environ['PATH'] = f"~/.conda/envs/env_MF_teledetection/bin:{os.environ['PATH']}"
os.environ['GDAL_DATA'] = '/opt/conda/env_MF_teledetection/share/gdal'
os.environ['PROJ_LIB'] = '/opt/conda/env_MF_teledetection/share/proj'

Les premiers fichiers TIF sont les suivants :  </br>
20260509_1500_meteosat12_europe_vis006.tif </br>
20260509_1500_meteosat12_europe_ir105.tif </br>
Il se trouve dans ce répertoire : **`/stockage/DATA/TIF/20260509`**</br></br>
Il s'agit d'un fichier type d'une production CMS envoyée par Transmet (extension .LT).

In [ ]:
input="/stockage/DATA/20260509"
VIS006="20260509_1500_meteosat12_europe_vis006"
IR105="20260509_1500_meteosat12_europe_ir105"
print(f"📁 Fichier source configuré : {VIS006}")
print(f"📁 Fichier source configuré : {IR105}")

<div class="alert alert-info alert-success">
<h3> 1 - 🔎 Visualisation du TIF et de son contenu</h3>
</div>

La commande `gdalinfo` affiche les métadonnées géospatiales d’un fichier raster. Dans Jupyter, on la précède de `!`.

In [ ]:
cd ~/MF_DATA_MANIPULATION

📁 **Création du dossier de résultats**

In [ ]:
!mkdir -p RESULTS
output = 'RESULTS'

## 🖼️ **Redimensionnement,conversion JPG et visualisation de l'image**

Nous allons redimensionner l'image et la convertir en JPG. L'utilisation de l'overview `[3]` (image 2784×2784 pixels) permet un gain de temps.

In [ ]:
{input}

In [ ]:
!convert   -resize 960 {input}/{VIS006}.tif  {output}/{VIS006}_min.jpg
chemin_image =f"{output}/{VIS006}_min.jpg"
im = Image.open(chemin_image, 'r')
display(im)

In [ ]:
!gdal_translate -scale 0 330 0 255 -ot Byte {input}/{IR105}.tif -outsize 960 540 {output}/{IR105}_min.tif -co COMPRESS=LZW
!convert  {output}/{IR105}_min.tif {output}/{IR105}_min.jpg  
chemin_image =f"{output}/{IR105}_min.jpg"
im = Image.open(chemin_image, 'r')
display(im)

In [ ]:
!gdaldem color-relief {input}/{IR105}.tif OUTILS/palette_sandwich_dn_transp.txt {output}/{IR105}_colorisee.tif -alpha
!convert -resize 960x540 {output}/{IR105}_colorisee.tif {output}/{IR105}_colorisee.jpg  
chemin_image =f"{output}/{IR105}_colorisee.jpg"
im = Image.open(chemin_image, 'r')
display(im)

option - alpha importante

In [ ]:
!gdalinfo {output}/{IR105}_colorisee.tif | grep "Band"

In [ ]:
# Conversion VIS en RGB 
!gdal_calc.py -A {input}/{VIS006}.tif --outfile={output}/red.tif --calc="A"
!gdal_calc.py -A {input}/{VIS006}.tif --outfile={output}/green.tif --calc="A"
!gdal_calc.py -A {input}/{VIS006}.tif --outfile={output}/blue.tif --calc="A"
!gdal_merge.py -separate -o {output}/{VIS006}_rgb.tif {output}/red.tif {output}/green.tif {output}/blue.tif

In [ ]:
#fusion simple

In [ ]:
from osgeo import gdal
import numpy as np
import os

output = "RESULTS"  # À ajuster​
VIS006_rgb = f"{output}/20260509_1500_meteosat12_europe_vis006_rgb.tif"
IR_colorisee = f"{output}/20260509_1500_meteosat12_europe_ir105_colorisee.tif"
fusion_sortie = f"{output}/fusion_finale.tif"

# Ouvrir les images
vis = gdal.Open(VIS006_rgb)
ir = gdal.Open(IR_colorisee)

# Lire les bandes
vis_r = vis.GetRasterBand(1).ReadAsArray()
vis_g = vis.GetRasterBand(2).ReadAsArray()
vis_b = vis.GetRasterBand(3).ReadAsArray()

ir_r = ir.GetRasterBand(1).ReadAsArray()
ir_g = ir.GetRasterBand(2).ReadAsArray()
ir_b = ir.GetRasterBand(3).ReadAsArray()
ir_a = ir.GetRasterBand(4).ReadAsArray()

# Normaliser alpha (0-255 -> 0-1)
alpha = ir_a / 255.0

# Fusion : IR par dessus VIS
fusion_r = (ir_r * alpha + vis_r * (1 - alpha)).astype(np.uint8)
fusion_g = (ir_g * alpha + vis_g * (1 - alpha)).astype(np.uint8)
fusion_b = (ir_b * alpha + vis_b * (1 - alpha)).astype(np.uint8)

# Sauvegarder
driver = gdal.GetDriverByName('GTiff')
out = driver.Create(fusion_sortie, vis.RasterXSize, vis.RasterYSize, 3, gdal.GDT_Byte)
out.SetProjection(vis.GetProjection())
out.SetGeoTransform(vis.GetGeoTransform())
out.GetRasterBand(1).WriteArray(fusion_r)
out.GetRasterBand(2).WriteArray(fusion_g)
out.GetRasterBand(3).WriteArray(fusion_b)
out = None

print(f"Fusion terminée : {fusion_sortie}")

# Afficher
from PIL import Image
import matplotlib.pyplot as plt

!convert -resize 3840 {output}/fusion_finale.tif {output}/fusion_finale_hd.jpg
!convert -resize 960 {output}/fusion_finale.tif {output}/fusion_finale_min.jpg

chemin_image =f"{output}/fusion_finale_min.jpg"
im = Image.open(chemin_image, 'r')
display(im)


In [ ]:
#version simplifiee multiplication , masque binaire

In [ ]:
from osgeo import gdal
import numpy as np

# Ouvrir les images
vis_ds = gdal.Open(f'{output}/{VIS006}_rgb.tif')
ir_ds = gdal.Open(f'{output}/{IR105}_colorisee.tif')

# Lire les données
vis_r = vis_ds.GetRasterBand(1).ReadAsArray().astype(np.float32)
vis_g = vis_ds.GetRasterBand(2).ReadAsArray().astype(np.float32)
vis_b = vis_ds.GetRasterBand(3).ReadAsArray().astype(np.float32)

ir_r = ir_ds.GetRasterBand(1).ReadAsArray().astype(np.float32)
ir_g = ir_ds.GetRasterBand(2).ReadAsArray().astype(np.float32)
ir_b = ir_ds.GetRasterBand(3).ReadAsArray().astype(np.float32)
ir_a = ir_ds.GetRasterBand(4).ReadAsArray().astype(np.float32)

# Créer un masque : 1 là où l'IR est opaque (alpha > 0), 0 ailleurs
mask = (ir_a > 0).astype(np.float32)

# Normalisation 0-255 -> 0-1
vis_r_norm = vis_r / 255.0
vis_g_norm = vis_g / 255.0
vis_b_norm = vis_b / 255.0

ir_r_norm = ir_r / 255.0
ir_g_norm = ir_g / 255.0
ir_b_norm = ir_b / 255.0

# Multiplication uniquement sur les zones du masque
# Ailleurs, on garde la VIS seule
result_r = (ir_r_norm * vis_r_norm * mask + vis_r_norm * (1 - mask)) * 255
result_g = (ir_g_norm * vis_g_norm * mask + vis_g_norm * (1 - mask)) * 255
result_b = (ir_b_norm * vis_b_norm * mask + vis_b_norm * (1 - mask)) * 255

# Clipper et convertir
result_r = np.clip(result_r, 0, 255).astype(np.uint8)
result_g = np.clip(result_g, 0, 255).astype(np.uint8)
result_b = np.clip(result_b, 0, 255).astype(np.uint8)

# Sauvegarder
driver = gdal.GetDriverByName('GTiff')
out_ds = driver.Create(f'{output}/sandwich_masque.tif', 
                       vis_ds.RasterXSize, vis_ds.RasterYSize, 3, gdal.GDT_Byte)
out_ds.SetProjection(vis_ds.GetProjection())
out_ds.SetGeoTransform(vis_ds.GetGeoTransform())
out_ds.GetRasterBand(1).WriteArray(result_r)
out_ds.GetRasterBand(2).WriteArray(result_g)
out_ds.GetRasterBand(3).WriteArray(result_b)
out_ds = None

# Visualiser
#gdal.Translate(f'{output}/sandwich_masque.png', f'{output}/sandwich_masque.tif', format='PNG')

print("Sandwich avec masque terminé !")

!convert -resize 3840 {output}/sandwich_masque.tif {output}/sandwich_masque_hd.jpg
!convert -resize 960 {output}/sandwich_masque.tif {output}/sandwich_masque_min.jpg

chemin_image =f"{output}/sandwich_masque_min.jpg"
im = Image.open(chemin_image, 'r')
display(im)

In [ ]:
# version sandwich avec transition douce

In [ ]:
from osgeo import gdal
import numpy as np

opacity = 65  # Force du sandwich (0 = seulement VIS, 100 = multiplication maximale)

# Ouvrir les images
vis_ds = gdal.Open(f'{output}/{VIS006}_rgb.tif')
ir_ds = gdal.Open(f'{output}/{IR105}_colorisee.tif')

# Lire les données
vis_r = vis_ds.GetRasterBand(1).ReadAsArray().astype(np.float32)
vis_g = vis_ds.GetRasterBand(2).ReadAsArray().astype(np.float32)
vis_b = vis_ds.GetRasterBand(3).ReadAsArray().astype(np.float32)

ir_r = ir_ds.GetRasterBand(1).ReadAsArray().astype(np.float32)
ir_g = ir_ds.GetRasterBand(2).ReadAsArray().astype(np.float32)
ir_b = ir_ds.GetRasterBand(3).ReadAsArray().astype(np.float32)
ir_a = ir_ds.GetRasterBand(4).ReadAsArray().astype(np.float32)

# Normalisation
vis_norm = vis_r / 255, vis_g / 255, vis_b / 255
ir_norm = ir_r / 255, ir_g / 255, ir_b / 255
alpha = ir_a / 255.0

# facteur = alpha * (opacity/100)
facteur = alpha * (opacity / 100.0)

# Fusion progressive
result_r = (ir_norm[0] * vis_norm[0] * facteur + vis_norm[0] * (1 - facteur)) * 255
result_g = (ir_norm[1] * vis_norm[1] * facteur + vis_norm[1] * (1 - facteur)) * 255
result_b = (ir_norm[2] * vis_norm[2] * facteur + vis_norm[2] * (1 - facteur)) * 255

# Clipper
result_r = np.clip(result_r, 0, 255).astype(np.uint8)
result_g = np.clip(result_g, 0, 255).astype(np.uint8)
result_b = np.clip(result_b, 0, 255).astype(np.uint8)

# Sauvegarder
driver = gdal.GetDriverByName('GTiff')
out_ds = driver.Create(f'{output}/sandwich_progressif_{opacity}.tif', 
                       vis_ds.RasterXSize, vis_ds.RasterYSize, 3, gdal.GDT_Byte)
out_ds.SetProjection(vis_ds.GetProjection())
out_ds.SetGeoTransform(vis_ds.GetGeoTransform())
out_ds.GetRasterBand(1).WriteArray(result_r)
out_ds.GetRasterBand(2).WriteArray(result_g)
out_ds.GetRasterBand(3).WriteArray(result_b)
out_ds = None

#gdal.Translate(f'{output}/sandwich_progressif_{opacity}.png', f'{output}/sandwich_progressif_{opacity}.tif', format='PNG')

print(f"Sandwich progressif avec facteur {opacity}% terminé !")


!convert -resize 3840 {output}/sandwich_progressif_{opacity}.tif {output}/sandwich_progressif_{opacity}_hd.jpg
!convert -resize 960 {output}/sandwich_progressif_{opacity}.tif {output}/sandwich_progressif_{opacity}_min.jpg

chemin_image =f"{output}/sandwich_progressif_{opacity}_min.jpg"
im = Image.open(chemin_image, 'r')
display(im)

Comparer les produits sur la page de la Galerie CMS en les téléchargeant (jpg)
http://e-sat.cms.meteo.fr/GALERIE/slider4_vierge.html

In [ ]:
teste avec le prduit Convection rgb

In [ ]:
Convection="20260509_1500_meteosat12_europe_ConvStorm"

In [ ]:
from osgeo import gdal
import numpy as np

def appliquer_masque(image_source, image_masque, image_sortie):
    """
    Applique le canal alpha d'une image masque à une image source RGB
    """
    # Ouvrir l'image source (celle qu'on veut masquer)
    src_ds = gdal.Open(image_source)
   
    # Ouvrir l'image masque (pour récupérer son alpha)
    mask_ds = gdal.Open(image_masque)
    alpha = mask_ds.GetRasterBand(4).ReadAsArray()  # Canal alpha
   
    # Lire les bandes RGB de la source
    r = src_ds.GetRasterBand(1).ReadAsArray()
    g = src_ds.GetRasterBand(2).ReadAsArray()
    b = src_ds.GetRasterBand(3).ReadAsArray()
   
    # Appliquer le masque (où alpha=0, on met 0)
    r_masked = np.where(alpha > 0, r, 0).astype(np.uint8)
    g_masked = np.where(alpha > 0, g, 0).astype(np.uint8)
    b_masked = np.where(alpha > 0, b, 0).astype(np.uint8)
   
    # Sauvegarder en RGBA
    driver = gdal.GetDriverByName('GTiff')
    out_ds = driver.Create(image_sortie,
                          src_ds.RasterXSize, src_ds.RasterYSize, 4, gdal.GDT_Byte)
    out_ds.SetProjection(src_ds.GetProjection())
    out_ds.SetGeoTransform(src_ds.GetGeoTransform())
    out_ds.GetRasterBand(1).WriteArray(r_masked)
    out_ds.GetRasterBand(2).WriteArray(g_masked)
    out_ds.GetRasterBand(3).WriteArray(b_masked)
    out_ds.GetRasterBand(4).WriteArray(alpha)  # Garder la transparence originale
    out_ds = None
   
    print(f"Masque appliqué : {image_sortie}")

# Utilisation
appliquer_masque(
    f'{input}/{Convection}.tif',
    f'{output}/{IR105}_colorisee.tif',
    f'{output}/nouvelle_image_masquee.tif'
) 

!convert -resize 3840 {output}/nouvelle_image_masquee.tif {output}/nouvelle_image_masquee_hd.jpg
!convert -resize 960 {output}/nouvelle_image_masquee.tif {output}/nouvelle_image_masquee_min.jpg

chemin_image =f"{output}/nouvelle_image_masquee_min.jpg"
im = Image.open(chemin_image, 'r')
display(im)


In [ ]:
teste sandwich avec vis et convection

In [ ]:
from osgeo import gdal
import numpy as np

opacity = 100  # Force du sandwich (0 = seulement VIS, 100 = multiplication maximale)

# Ouvrir les images
vis_ds = gdal.Open(f'{output}/{VIS006}_rgb.tif')
ir_ds = gdal.Open(f'{output}/nouvelle_image_masquee.tif')

# Lire les données
vis_r = vis_ds.GetRasterBand(1).ReadAsArray().astype(np.float32)
vis_g = vis_ds.GetRasterBand(2).ReadAsArray().astype(np.float32)
vis_b = vis_ds.GetRasterBand(3).ReadAsArray().astype(np.float32)

ir_r = ir_ds.GetRasterBand(1).ReadAsArray().astype(np.float32)
ir_g = ir_ds.GetRasterBand(2).ReadAsArray().astype(np.float32)
ir_b = ir_ds.GetRasterBand(3).ReadAsArray().astype(np.float32)
ir_a = ir_ds.GetRasterBand(4).ReadAsArray().astype(np.float32)

# Normalisation
vis_norm = vis_r / 255, vis_g / 255, vis_b / 255
ir_norm = ir_r / 255, ir_g / 255, ir_b / 255
alpha = ir_a / 255.0

# facteur = alpha * (opacity/100)
facteur = alpha * (opacity / 100.0)

# Fusion progressive
result_r = (ir_norm[0] * vis_norm[0] * facteur + vis_norm[0] * (1 - facteur)) * 255
result_g = (ir_norm[1] * vis_norm[1] * facteur + vis_norm[1] * (1 - facteur)) * 255
result_b = (ir_norm[2] * vis_norm[2] * facteur + vis_norm[2] * (1 - facteur)) * 255

# Clipper
result_r = np.clip(result_r, 0, 255).astype(np.uint8)
result_g = np.clip(result_g, 0, 255).astype(np.uint8)
result_b = np.clip(result_b, 0, 255).astype(np.uint8)

# Sauvegarder
driver = gdal.GetDriverByName('GTiff')
out_ds = driver.Create(f'{output}/sandwich_progressif_{opacity}_test.tif', 
                       vis_ds.RasterXSize, vis_ds.RasterYSize, 3, gdal.GDT_Byte)
out_ds.SetProjection(vis_ds.GetProjection())
out_ds.SetGeoTransform(vis_ds.GetGeoTransform())
out_ds.GetRasterBand(1).WriteArray(result_r)
out_ds.GetRasterBand(2).WriteArray(result_g)
out_ds.GetRasterBand(3).WriteArray(result_b)
out_ds = None

#gdal.Translate(f'{output}/sandwich_progressif_{opacity}.png', f'{output}/sandwich_progressif_{opacity}.tif', format='PNG')

print(f"Sandwich progressif avec facteur {opacity}% terminé !")


!convert -resize 3840 {output}/sandwich_progressif_{opacity}_test.tif {output}/sandwich_progressif_{opacity}_test_hd.jpg
!convert -resize 960 {output}/sandwich_progressif_{opacity}_test.tif {output}/sandwich_progressif_{opacity}_test_min.jpg

chemin_image =f"{output}/sandwich_progressif_{opacity}_test_min.jpg"
im = Image.open(chemin_image, 'r')
display(im)

### Pansharpening

In [ ]:
Convection="20260509_1500_meteosat12_europe_ConvStorm"

In [ ]:
!gdalinfo {input}/{VIS006}.tif |grep "Size is"
!gdalinfo {input}/{Convection}.tif |grep "Size is"

In [ ]:
!gdal_pansharpen.py {input}/{VIS006}.tif {input}/{Convection}.tif  {output}/{Convection}_vis_pansharp.tif
!convert -resize 3840 {output}/{Convection}_vis_pansharp.tif {output}/{Convection}_vis_pansharp_hd.jpg
!convert -resize 960 {output}/{Convection}_vis_pansharp.tif {output}/{Convection}_vis_pansharp_min.jpg

!convert -resize 3840 {input}/{Convection}.tif {output}/{Convection}_hd.jpg
!convert -resize 960 {input}/{Convection}.tif {output}/{Convection}_min.jpg

chemin_image_convection =f"{output}/{Convection}_min.jpg"
im_convection = Image.open(chemin_image_convection, 'r')
display(im_convection)

chemin_image =f"{output}/{Convection}_vis_pansharp_min.jpg"
im = Image.open(chemin_image, 'r')
display(im)

In [ ]:
#pansharpening avec l'ir colorisée (juste partie colorisée)

In [ ]:
!gdal_pansharpen.py {input}/{VIS006}.tif {output}/{IR105}_colorisee.tif -r cubic {output}/{VIS006}_ir_pansharpening.tif

!convert -resize 3840 {output}/{VIS006}_ir_pansharpening.tif {output}/{VIS006}_ir_pansharpening_hd.jpg
!convert -resize 960 {output}/{VIS006}_ir_pansharpening.tif {output}/{VIS006}_ir_pansharpening_min.jpg

chemin_image =f"{output}/{VIS006}_ir_pansharpening_min.jpg"
im = Image.open(chemin_image, 'r')
display(im)

📌 **Description** : Image **Geocolor RGB** en vue géostationnaire (GEOS), depuis le satellite **Meteosat-12** positionné à 0° de longitude. Ce produit est réalisé à partir de fichiers multicanaux, avec diverses corrections (Diffusion, parallaxe, correction angle solaire).

> 💡 *Le produit Geocolor RGB combine plusieurs canaux pour restituer une image en "vraies couleurs" jour et nuit.*

<div class="alert alert-info alert-success">
<h3> 2 - 🌐 Comparaison des formats d'image</h3>
</div>

| Format | Compression | Poids | Géoréférencement | Utilisation |
|--------|-------------|-------|------------------|-------------|
| **JPG** | Avec pertes (défaut 92) | Léger | ❌ Non | Web, visualisation rapide |
| **PNG** | Sans pertes | Lourd (4× JPG) | ❌ Non | Archives, transparence |
| **TIF** | Optionnelle (LZW, JPEG) | Très lourd (7× JPG) | ✅ Oui | SIG, traitement géospatial |

> ⚠️ *Seul le format TIF conserve le géoréférencement (coordonnées, projection).*</br>
> ⚠️ *Même avec une compression à 100, le format JPEG crée une altération.*</br>
> *Les différences visuelles sont souvent très ténues, voire invisibles, l'interpolation JPG par défaut "Lanczos" est un très bon compromis netteté / aliasing (pixellisation visible).*

In [ ]:
!convert -resize 700x700 $INPUT[3]  $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.jpg  >/dev/null 2>&1 
!convert -resize 700x700 $INPUT[3]  $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif  >/dev/null 2>&1 
!convert -resize 700x700 $INPUT[3]  $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.png  >/dev/null 2>&1 
!identify -quiet -format "%f : %b\n" $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.png

📊 **Observations** :
- Le format **PNG** est environ **4 fois plus lourd** que le JPG
- Le format **TIF** est environ **7 fois plus lourd** que le JPG

❗ **Attention** : La commande `convert` (ImageMagick) **ne conserve pas le géoréférencement**.

In [ ]:
!gdalinfo $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif 2>/dev/null | head -20

### 🔧 **Pour conserver le géoréférencement : `gdalwarp`**

La commande **`gdalwarp`** redimensionne l'image tout en préservant les informations géospatiales.

In [ ]:
!gdalwarp -overwrite -ts 700 700 $INPUT  $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif  >/dev/null 2>&1

In [ ]:
!gdalinfo $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif 2>/dev/null | head -25

✅ **Succès** : Les métadonnées de géoréférencement sont bien conservées.

### 🗜️ **Compression des fichiers TIF**

| Compression | Type | Gain | Usage |
|-------------|------|------|-------|
| **LZW** | Sans perte | ~25% | Archives, données précises |
| **JPEG/YCbCr** | Avec pertes (qualité 95) | ~80% | Diffusion, web, visualisation |

> 💡 *La compression YCbCr JPEG95 (Luminance/couleur) est très efficace. Le gain en poids est important, mais il y a des pertes (infimes, souvent invisibles, mais existantes)*</br>

In [ ]:
!gdalwarp -overwrite -ts 700 700 $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_compress_YCbCr.tif -co COMPRESS=JPEG -co JPEG_QUALITY=95 -co PHOTOMETRIC=YCBCR >/dev/null 2>&1
!gdalwarp -overwrite -ts 700 700 $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_compress_LZW.tif -co COMPRESS=LZW -co PREDICTOR=2  >/dev/null 2>&1
!identify -quiet -format "%f : %b\n" $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif
!identify -quiet -format "%f : %b\n" $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_compress_YCbCr.tif
!identify -quiet -format "%f : %b\n" $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_compress_LZW.tif

📊 **Résultats** :
- **LZW** : Poids ≈ 75% du fichier non compressé
- **YCbCr (JPEG)** : Poids ≈ 20% du fichier non compressé

> 👉 La compression est un paramètre très important à prendre en compte, que cela soit en ce qui concerne la perte d'information, ou le volume à manipuler.

> 💡 *Une image Meteosat-12 plein disque à 500 m en TIF compressé YCbCr pèse environ **160 Mo**, contre **1,4 Go** non compressée.*

<div class="alert alert-info alert-success">
<h3> 3 - 🌐 Reprojections</h3>
</div>

`gdalwarp` permet de changer la projection d'une image.

### 🗺️ **Projection EPSG:4326 (WGS84 - longitude/latitude)**

Système de coordonnées géographiques en degrés. Visuellement : **cylindrique équidistante** (planisphère "plate").</br>
*WGS : World Geodetic System 1984,*</br>
*EPSG:4326 : Système géographique*

In [ ]:
!gdalwarp -overwrite -t_srs EPSG:4326 $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_4326.tif >/dev/null 2>&1
!gdalwarp -overwrite -t_srs EPSG:4326 $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_4326.jpg >/dev/null 2>&1

In [ ]:
!gdalinfo $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_4326.tif 2>/dev/null | head -30

On retrouve le système  WGS 84, </br>
Les coordonnées en latitude / longitude (degrés)

In [ ]:
im2 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_4326.jpg', 'r')
display(im2)

---

### 🧭 **Projection Mercator (EPSG:3395)**

Projection cylindrique **conforme** (conservation des angles). </br>
Référence dans le monde grâce à sa précision pour les voyages marins du fait de la conservation des angles qu'elle permet, essentielle pour prévoir les routes de navigation.

| EPSG | Nom | Utilisation |
|------|-----|-------------|
| 3395 | Mercator (ellipsoïde) | Données précises |
| 3857 | Web Mercator (sphérique) | Google Maps, OpenStreetMap |

In [ ]:
!gdalwarp -overwrite -t_srs EPSG:3395 $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_3395.jpg >/dev/null 2>&1

In [ ]:
im3 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_3395.jpg', 'r')
display(im3)

> 💡 *On peut aussi utiliser `-t_srs '+proj=merc'` comme alternative à `EPSG:3395`.*

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=merc' $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_merc.jpg >/dev/null 2>&1

In [ ]:
im3bis = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_merc.jpg', 'r')
display(im3bis)

> ℹ️ *Les fichiers `merc.jpg` et `4326.jpg` sont identiques car la projection Mercator à l'échelle globale avec ces paramètres donne un résultat visuellement similaire à WGS84.*

In [ ]:
!diff $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_merc.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_4326.jpg
!diff $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_merc.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_3395.jpg

---

### 🌍 **Projection de Peters (Gall-Peters)**

Projection cylindrique équivalente qui respecte les **superficies réelles** des continents (contrairement à Mercator).

In [ ]:
!gdalwarp -overwrite -t_srs "+proj=cea +lon_0=0 +lat_ts=45 +x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs" $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_peter.jpg >/dev/null 2>&1

In [ ]:
im3 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_peter.jpg', 'r')
display(im3)

<div class="alert alert-info alert-success">
<h3> 4 - ✂️ Découpes (cropping)</h3>
</div>

### 🇫🇷 **Découpe sur la France (projection EPSG:4326)**

L'avantage de la projection WGS84 est de pouvoir spécifier les limites en degrés.

**Options importantes** :
- `-t_srs {proj}` : nouvelle projection
- `-te {west} {south} {east} {north}` : étendue de découpe (W S E N)
- `-ts {width} {height}` : taille de sortie en pixels
- `-tr {xres} {yres}` : résolution spatiale

In [ ]:
!time gdalwarp -overwrite -t_srs EPSG:4326 -te -14 38 18 56 -ts 960 540 $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_4326_france.tif
!time convert $output/globeMtruecolorGTP.mtg.20260422.1200.24_4326_france.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_4326_france.jpg

In [ ]:
!time gdalwarp -overwrite -t_srs EPSG:4326 -te -14 38 18 56 -ts 960 540 $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_4326_france.jpg >/dev/null 2>&1

Cette commande permet de découper/reprojeter directement le résultat. 
> ⚠️ Une autre possibilité aurait été de reprojeter le plein disque, et de découper ensuite, mais cette approche est environ **40x plus longue** !

In [ ]:
#!time gdalwarp -overwrite -t_srs EPSG:4326 $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_4326.tif 
#!gdalwarp -overwrite -te -14 38 18 56 -ts 960 540  $output/globeMtruecolorGTP.mtg.20260422.1200.24_4326.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_4326_france.jpg #W S E N en projection 16/9 

In [ ]:
im4 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_4326_france.jpg', 'r')
display(im4)

> ⚠️ **Attention** : L'image peut être déformée si le ratio de sortie (`-ts`) ne correspond pas au ratio géographique de la zone découpée.</br>
> Ici, la taille 960x540 correspond a du 16/9.

### **Reprojection en Stéréo polaire sur la France (projection EPSG:3995)**

La projection stéréographique polaire est une projection cartographique qui représente la Terre vue depuis un pôle sur un plan tangent au pôle. Elle conserve bien les angles et les formes locales, mais déforme les surfaces et distances loin du pôle

In [ ]:
!gdalwarp -overwrite -t_srs EPSG:3995 -ts 960 540 $output/globeMtruecolorGTP.mtg.20260422.1200.24_4326_france.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_stereo_france.jpg >/dev/null 2>&1
im_stereo = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_stereo_france.jpg', 'r')
display(im_stereo)

### 🛰️ **Projection orthographique (vue de l'espace)**

La projection orthographique donne l'impression de regarder la Terre depuis l'espace, à la verticale d'un point.

**Paramètres** :
- `lat_0` : latitude du centre
- `lon_0` : longitude du centre
- `-te` : étendue en mètres (optionnel)

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -ts 960 540 -te -1120000 -630000 1120000 630000 -r cubic $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1.jpg >/dev/null 2>&1
im6 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1.jpg', 'r')
display(im6)

📏 **Résolution** : Dans cet exemple, la résolution est d'environ 2333 m/pixel (perte par rapport à l'original).
-> Nous avons donc perdu en résolution spatiale

### 🔍 **Sans imposer la taille (`-ts`)**

La pleine résolution est conservée.

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -te -1120000 -630000 1120000 630000 -r cubic $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_HD.jpg >/dev/null 2>&1

### 🎚️ **Imposer une résolution spatiale avec `-tr`**

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -tr 750 750 -te -1120000 -630000 1120000 630000 -r cubic $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_tr.jpg >/dev/null 2>&1
im6 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_tr.jpg', 'r')
display(im6)

### 🎨 **Méthodes de rééchantillonnage (`-r`)**

| Option | Description | Usage |
|--------|-------------|-------|
| `near` | Plus proche voisin | Rapide, préserve les valeurs (données catégorielles) |
| `bilinear` | Bilinéaire | Lissage modéré, bon compromis |
| `cubic` | Cubique (défaut recommandé) | Lisse et net, idéal pour images |
| `lanczos` | Lanczos | Très net mais plus lent, peut créer des artefacts |

> 💡 **Recommandation** : Utilisez `-r cubic` pour un bon compromis entre netteté et lissage.

### 🔍 **Comparaison des images : zoom sur le nord Alpes**

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.6 lon_0=7.3' -te -112000 -63000 112000 63000 $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom.jpg
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.6 lon_0=7.3' -tr 750 750 -te -112000 -63000 112000 63000 $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_tr_zoom.jpg
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.6  lon_0=7.3' -ts 100 56 -te -112000 -63000 112000 63000 $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_ts_zoom.jpg
im7 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom.jpg', 'r')
print("🏭 option -te")
display(im7)
im8 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_tr_zoom.jpg', 'r')
print("🏭 option -tr")
display(im8)
im9 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_ts_zoom.jpg', 'r')
print("🏭 option -ts")
display(im9)

* Agrandissement des images :

In [ ]:
!convert -quiet -resize 675x380! $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom.jpg  $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom_675x380.jpg >/dev/null 2>&1
!convert -quiet -resize 675x380! $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_tr_zoom.jpg  $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_tr_zoom_675x380.jpg >/dev/null 2>&1
!convert -quiet -resize 675x380! $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_ts_zoom.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_ts_zoom_675x380.jpg >/dev/null 2>&1
im10 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom_675x380.jpg', 'r')
print("🏭 option -te")
display(im10)
im11 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_tr_zoom_675x380.jpg', 'r')
print("🏭 option -tr")
display(im11)
im12 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_ts_zoom_675x380.jpg', 'r')
print("🏭 option -ts")
display(im12)

* Exemples de différences de méthode de rééchantillonage : avec et sans -r cubic

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.6 lon_0=7.3'  -ts 1350 760 -te -56000 -31500 56000 31500 -r cubic $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom_1350x760-rcubic.jpg
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.6 lon_0=7.3'  -ts 1350 760 -te -56000 -31500 56000 31500 $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom_1350x760.jpg
im13 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom_1350x760-rcubic.jpg', 'r')
display(im13)
im14 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom_1350x760.jpg', 'r')
display(im14)

## 🎨 **Visualisation des bandes RGB**

On peut extraire chaque bande (Rouge, Vert, Bleu) d'une image RGB.

In [ ]:
!gdal_translate -b 1 $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_bandeR.jpg >/dev/null 2>&1
!gdal_translate -b 2 $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_bandeG.jpg >/dev/null 2>&1
!gdal_translate -b 3 $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_bandeB.jpg >/dev/null 2>&1

In [ ]:
!convert -resize 300x300 $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_bandeR.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_min_bandeR.jpg >/dev/null 2>&1
!convert -resize 300x300 $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_bandeG.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_min_bandeG.jpg >/dev/null 2>&1
!convert -resize 300x300 $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_bandeB.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_min_bandeB.jpg >/dev/null 2>&1

In [ ]:
im15 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_min_bandeR.jpg', 'r')
print("🏭 Bande Rouge")
display(im15)
im16 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_min_bandeG.jpg', 'r')
print("🏭 Bande Verte")
display(im16)
im17 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_min_bandeB.jpg', 'r')
print("🏭 Bande Bleue")
display(im17)

📚 **Pour aller plus loin** :
- 📌 [Fiche technique Geocolor RGB](../DOCS/MTG_fiche_geocolor_RGB.pdf)
- 📌 [Recettes des produits RGB](../DOCS/mtg_fci_synthese_produits.pdf)
- 🛠️ [Visualisation interactive des bandes](http://e-sat.cms.meteo.fr/GALERIE/loupes.html)

<div class="alert alert-info alert-success">
<h3> 5 - 🖌️ Ajouter un contour, un logo, annoter l'image</h3>
</div>

### 🗺️ **Ajout de contours avec `gdal_rasterize`**

La commande `gdal_rasterize` permet de dessiner des contours (fichiers shapefile) sur une image.

**Options de couleur** :
- `-b 1 -burn 255 -b 2 -burn 255 -b 3 -burn 255` → blanc
- `-b 1 -burn 0 -b 2 -burn 0 -b 3 -burn 0` → noir
- `-b 1 -burn 255 -b 2 -burn 0 -b 3 -burn 0` → rouge

> ⚠️ *Attention : `gdal_rasterize` modifie directement le fichier d'entrée (le TIF est écrasé).*

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -ts 960 540 -te -1120000 -630000 1120000 630000 -r cubic $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1.tif
!gdal_rasterize -b 1 -burn 200 -b 2 -burn 200 -b 3 -burn 200 -l world-administrative-boundaries OUTILS/boundary/world-administrative-boundaries.shp  $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1.tif >/dev/null 2>&1
!convert -quiet -resize 1920x1080 $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour.jpg
im18 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour.jpg', 'r')
display(im18)

### 📝 **Ajout d'un titre et d'un bandeau**

In [ ]:
titre="Meteosat-12 - Geocolor RGB - 22/04/2026 à 12 h UTC"
# Ajout d'un bandeau bleu semi-transparent
!composite -quiet -geometry +0+0 OUTILS/bandeau_bleu_1920x1080.png $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour_bandeau.jpg
# Ajout du texte
!convert $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour_bandeau.jpg -font OUTILS/Police_Marianne/Marianne-Bold.otf -pointsize 25 -fill '#eeeeee' -annotate +24+24 "$titre" $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour_date.jpg >/dev/null 2>&1
im19 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour_date.jpg', 'r')
display(im19)

### 🏷️ **Ajout du logo Météo-France**

In [ ]:
!composite -quiet -geometry +1520+1016 OUTILS/logo_MF-400.png $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour_date.jpg $output/globeMtruecolor.mtg.20260422.1200_france.jpg
im20 = Image.open(output + '/globeMtruecolor.mtg.20260422.1200_france.jpg', 'r')
display(im20)

<div class="alert alert-info alert-success">
<h3> 6 - 🎬 Création d'une animation</h3>
</div>

### 🔁 **Boucle sur les 24 heures**

L'exemple suivant génère une animation pour les 24 heures de la journée du 22 avril 2026.

In [ ]:
%%bash
output="RESULTS"
mkdir -p $output/tmp
rm -f  $output/tmp/*
for hh in {00..23}
do
    echo "Traitement de l'heure : ${hh}h00"
    INPUT="/stockage/DATA/TIF/20260422/EOXF71MTGG22${hh}00.globeMtruecolorGTP.mtg.20260422.${hh}00.*.tif.LT"
    # Découpe et reprojection
    gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -te -1120000 -630000 1120000 630000 -ts 1920 1080 -r cubic $INPUT $output/tmp/geocolor.mtg.20260422_${hh}00_france.tif >/dev/null 2>&1
    # Ajout des contours
    gdal_rasterize -b 1 -burn 200 -b 2 -burn 200 -b 3 -burn 200 -l world-administrative-boundaries OUTILS/boundary/world-administrative-boundaries.shp $output/tmp/geocolor.mtg.20260422_${hh}00_france.tif >/dev/null 2>&1
    # Ajout du bandeau
    composite -quiet -geometry +0+0 OUTILS/bandeau_bleu_1920x1080.png $output/tmp/geocolor.mtg.20260422_${hh}00_france.tif $output/tmp/geocolor.mtg.20260422_${hh}00_france_contour_bandeau.tif >/dev/null 2>&1
    # Ajout logo
    composite -quiet -geometry +1520+1016 OUTILS/logo_MF-400.png $output/tmp/geocolor.mtg.20260422_${hh}00_france_contour_bandeau.tif $output/tmp/geocolor.mtg.20260422_${hh}00_france_contour_bandeau_logo.tif
    # Ajout du titre
    titre="Meteosat-12 - Geocolor RGB - 22/04/2026 à ${hh} h UTC"
    convert $output/tmp/geocolor.mtg.20260422_${hh}00_france_contour_bandeau_logo.tif -font OUTILS/Police_Marianne/Marianne-Bold.otf -pointsize 25 -fill '#eeeeee' -annotate +24+24 "$titre" $output/tmp/geocolor.mtg.20260422_${hh}00_france.jpg >/dev/null 2>&1
done
rm -f $output/tmp/*.tif

In [ ]:
# Création de l'animation avec ffmpeg
!ffmpeg -y -framerate 3 -pattern_type glob -i "$output/tmp/geocolor.mtg.20260422_*00_france.jpg" -pix_fmt yuv420p -vcodec libx264 -crf 27 $output/geocolor_mtg_20260422_france.mp4 >/dev/null 2>&1
!echo "✅ Animation créée : $output/geocolor_mtg_20260422_france.mp4"

## 🌟 **Ressources complémentaires**

| Lien | Description |
|------|-------------|
| [Galerie CMS](http://e-sat.cms.meteo.fr/GALERIE/archive.html) | Archives d'images satellitaires |
| [Animations temps réel](http://e-sat.cms.meteo.fr/GALERIE/animations.html) | Animations Meteosat en direct |
| [Documentation GDAL](https://gdal.org/en/stable/) | Référence complète de GDAL |

---

✅ **Fin du TP**